# Dunnhumby · NGCF · M1~M5_D 8모형 / seed49

확정된 **M2 임베딩·M3 중심화 N/V 그래프·M4 고객 내 평균1 손실 가중치**를 유지하고 백본만 NGCF로 교체합니다.

|순서|모형|M2 표현|M3 그래프|M4 손실|
|---|---|---|---|---|
|1|M1|없음|이진|무가중|
|2|M2|적응형 N/V|이진|무가중|
|3|M3|없음|중심화 N/V|무가중|
|4|M4|없음|이진|고객 내 평균1|
|5|M5_A|적응형 N/V|중심화 N/V|무가중|
|6|M5_B|적응형 N/V|이진|고객 내 평균1|
|7|M5_C|없음|중심화 N/V|고객 내 평균1|
|8|M5_D|적응형 N/V|중심화 N/V|고객 내 평균1|

- **새 NGCF 8개를 하나의 L4에서 순서대로 학습**합니다. 기존 LightGCN 성능/체크포인트를 NGCF 결과로 재사용하지 않습니다. 기존 참조는 확정 입력·가중치 검증에만 사용합니다.
- train DAY1~704 / test DAY705~711, 신규상품 추천(학습 구매쌍 제외), MIN_ITEM_INTER=1, uniform 음성1개.
- **각300epoch 마지막 체크포인트만 평가**. validation·holdout·조기종료·중간 test·자동 추가 시드·순열 학습 없음.
- CLV는 기존과 같은 학습구간 내365일 historical proxy. N/V·신뢰도·RBF·양성상품의 현재 구매자 제외·M2 적응형 계수/직교결합·M3/M4 계산을 유지합니다.
- NGCF: ID64, 전파2층(각64), 층 연결192차원, LeakyReLU(.2), message dropout .1, node dropout 없음. M2 인코더 출력만64→192로 맞춥니다.
- 공통 batch8192, Adam lr.0005, ID L2 .001, M2 공유표현 L2 1e-6. NGCF 변환행렬/편향 L2 1e-6. 모든 모형의 ID/NGCF 초기값을 동일 seed로 맞추고 같은 모델 안에서는 하나의 optimizer로 공동학습합니다.
- NGCF 논문의 자기 메시지·이웃 메시지·요소곱 상호작용과 층 연결을 적용하되, 프로젝트의 이진/가중 인접행렬·초기화·정규화 규칙을 사용한 **조건통제 구현**입니다. 원저자 벤치마크 수치의 정확한 재현을 주장하지 않습니다.
- **1시드의 기술적 비교**입니다. 10시드 평균·통계적 유의성·CLV 배정 귀속·새 독립 확증을 주장하지 않습니다. test를 보고 계수/epoch를 바꾸지 않습니다.

참고: [NGCF 논문](https://arxiv.org/abs/1905.08108), [저자 구현](https://github.com/xiangwang1223/neural_graph_collaborative_filtering).

## 중단 후 이어서 실행
같은 노트북·코드 버전·참조/출력 경로로 위에서부터 다시 실행하세요. **완료 모형은 건너뛰고, 미완료 모형은 마지막 저장 epoch 다음부터 재개**합니다. 모델·optimizer·난수 상태를 매 epoch 저장합니다. 중간에 끊긴 epoch만 다시 계산할 수 있습니다. 손상된 checkpoint나 Drive 오류가 있으면 파일을 지우거나 자동 처음부터 학습하지 않고 멈춥니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = 'dcbd7bc1b655d51945f0626e23ee9f79cbac828c'
REPO = Path('/content/clv-dh-ngcf-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 확정 CLV 입력과 기존 참조 검증 — 학습 없음
기존 Dunnhumby M2 10시드 및 M4 10시드 폴더에서 원본 보고서의 해시·분할·입력·M2 특징·M3/M4 가중치 정합성을 확인합니다. 파일이 맞지 않으면 학습 전에 중단합니다. **이 단계에서 기존 모델을 새로 학습하지 않습니다.** 재사용 준비 모듈의 A/B 안내가 나타나더라도 실제 실행 목록은 아래 NGCF 8개입니다.

기본 참조 폴더가 이동/변경되었다면 `m2_reference`, `m4_reference`에 보관 원본 ZIP 또는 폴더의 Drive 경로를 넣으세요. 첫 실행 이후에는 경로를 유지하세요. 출력은 별도 NGCF 폴더입니다.

In [ ]:
import clv_dh_ngcf_final8_seed49 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/원본M2결과.zip',
    # m4_reference='/content/drive/MyDrive/clv_m4_user_centered_final10_results.zip',
)
print('M2 입력 검증 참조:', cfg.m2_reference)
print('M3/M4 입력 검증 참조:', cfg.m4_reference)
print('NGCF 새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('실제 새 학습:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])

## 2. NGCF 8개 모형 순차 학습
5epoch마다 저장 안내, 25epoch마다 손실을 표시합니다. 모형 하나가300epoch에 도달하면 전체·CLV 세그먼트·노출 지표를 평가하고 결과를 즉시 Drive에 저장한 뒤 다음 모형으로 넘어갑니다. **NGCF M1도 새로 학습해야 합니다.**

학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요. 출력이 사라져도 중복 학습을 시작하기 전에 연결/현재 실행 상태를 확인하세요. 완료 모형의 결과는 모두 끝날 때까지 기다리지 않고 아래 결과 셀로 회수할 수 있습니다(학습 셀을 중단한 후 실행).

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 NGCF M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('NGCF M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/dh_ngcf_final8_seed49_results.zip')
print(output)
files.download(output)